# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)

In [ ]:
#show all columns
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

#dont show col types        
options(readr.show_col_types = FALSE)

## 0. Plot parameters

In [ ]:
parent_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setB_T2 "
parent_dir_2 = "/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setB_T3"
sample_sheet_csv = "/ceph.groups/mshahbazi.grp/rsakata/EXP62/sample_sheet.csv"
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/EXP62/output/setB_T2/4_plots"

EXP = "EXP62"
EXP_SUB = "setB_T2"

In [ ]:
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.3 

settheme <- theme_minimal() +
  theme(
    text = element_text(family = "sans", size = FONT.SIZE),
    panel.background = element_blank(),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"),
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(colour = "black"),
    axis.text = element_text(colour = "black"),
    axis.text.x = element_text(colour = "black", angle = 0),
    legend.position = "right",
    title = element_text(colour = "black", size = FONT.SIZE),
    plot.title = element_text(size = FONT.SIZE, face = "plain")
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F62", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_cell_cell = c("WT_WT" = "#5E5E5E", 
               "GFP_GFP" = "#86AB30", 
               "GFP_WT"= "#E2A557")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")
col_state = c("Developed"= "#86AB30","Failed"="#8d8d8dff")


## 1. Extract summary files

In [ ]:
files <- list.files(
  path = c(parent_dir,parent_dir_2) , 
  pattern = "_data\\.csv$", 
  recursive = TRUE, 
  full.names = TRUE
)

data_merged <- files %>%
  set_names() %>%
  map_dfr(read_csv, .id = "filename")

# add columns to identify the samples and image
data_merged <- data_merged %>%
  mutate(

    # Remove "_data.csv" from the end to get "image" ("10_1")
    image = sub("_data\\.csv$", "", basename(filename)),
    
    # Remove everything from the first "_" onwards to get "sample" ("10")
    sample = sub("_.*", "", image)
  )

data_merged$exp = EXP
data_merged$exp_sub = EXP_SUB

head(data_merged)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- data_merged %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

## Plots

### A) Normalised intensity (mean of mean per image)

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  # mean of each image
  group_by(sample_name, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE)
  ) 

In [ ]:
w <- 6
h <- 4
title = "Normalised_intensity_meanperimage"
options(repr.plot.width=w, repr.plot.height=h)

p <- ggplot(plot_data, aes(x = distance_from_peak, y = mean_int_1, color = sample_name, group = image )) +
  
  # The main average line
  geom_line(linewidth = 0.1) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid(channel~sample_name, scales = "free_y") +
  settheme 
ggsave(plot = p, filename = sprintf("%s/A1_%s.pdf",out_dir, title), w = w, h = h)
p

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  # mean of each image
  group_by(sample_name, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # mean of each sample
  group_by(sample_name, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%
  # Define the upper and lower limits for the shaded ribbon
  mutate(
    ymin = mean_int - se_int, # Change 'se_int' to 'sd_int' if you prefer Standard Deviation
    ymax = mean_int + se_int
  )%>%
  filter (channel %in% c("DAPI", "ECAD", "phalloidin", "GFP"))

In [ ]:
w <- 5
h <- 1.5
title = "Normalised_intensity_meanofmean"
options(repr.plot.width=w, repr.plot.height=h)

p <- ggplot(plot_data, aes(x = distance_from_peak, y = mean_int, fill = sample_name, color = sample_name)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_wrap(~ channel, nrow = 1) +
  settheme 
ggsave(plot = p, filename = sprintf("%s/A2_%s.pdf",out_dir, title), w = w, h = h)
p

### B) Background substracted normalised background 

In [ ]:
head(merged_df)

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(sample_name, condition_2, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each sample
  group_by(sample_name, condition_2, state ,distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(sample_name, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 6
h <- 2
title = "Normalised_intensity-background"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = sample_name, color = sample_name)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid( channel ~ sample_name, scales = "free_y") +
  settheme 

ggsave(plot = plot , filename = sprintf("%s/B_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# failed vs developed
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(sample_name, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each state
  group_by(state ,distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(state, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 4
h <- 1.5
title = "failed_developed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = state, color = state)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme + 
  scale_fill_manual(values = col_state)+ 
  scale_color_manual(values = col_state)

ggsave(plot = plot , filename = sprintf("%s/B2_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# by condition and state
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(sample_name, condition_2, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each state
  group_by(condition_2 ,state, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(condition_2, state, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 4
h <- 2
title = "condition_state"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = condition_2, color = condition_2)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid( state~ channel, scales = "free_y") +
  settheme + 
  scale_fill_manual(values = col_condition_2)+ 
  scale_color_manual(values = col_condition_2)

ggsave(plot = plot , filename = sprintf("%s/B3_%s.pdf",out_dir, title), w = w, h = h)
plot 

### C) Bar with dots

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(sample_name, condition, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(sample_name, condition, state, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 4
h <- 2
title = "peakintensity"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = sample_name, y = mean_int_1)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = condition),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0))+
  labs(
    title =title,
    y = "",
    x = ""
  )+ facet_wrap(~channel, nrow= 1) + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  ) + 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)
              

ggsave(plot = p  , filename = sprintf("%s/C_%s.pdf",out_dir, title), w = w, h = h)
p 

### D) GFP intensity

In [ ]:
gfp_df <- merged_df %>% 
  filter(channel == "GFP")


pair_stats <- gfp_df %>%
  group_by(sample_name, condition, image, z, label_i, label_j) %>%
  summarise(
    intensity_before = mean(normalized_intensity[distance_from_peak < 0], na.rm = TRUE),
    intensity_after  = mean(normalized_intensity[distance_from_peak > 0], na.rm = TRUE),
    .groups = "drop"
    )

plot_data <- pair_stats %>%
  pivot_longer(cols = c(intensity_before, intensity_after), 
                names_to = "position", values_to = "intensity") 

In [ ]:
w <- 2
h <- 4
title = "gfp_intensity"
options(repr.plot.width=w, repr.plot.height=h)

GFP_thresh = 1


  p <- ggplot(plot_data, aes(x = intensity, fill = condition)) +
    geom_histogram(bins = 50, alpha = 0.5, position = "identity") +
    #scale_fill_manual(values = c(intensity_before = "blue", intensity_after = "orange")) +
    labs(title = "GFP Intensity Distribution", fill = "Group") +
    facet_wrap(~sample_name, ncol=1, scales = "free_y")+ 
    geom_vline(xintercept = GFP_thresh,
              linetype = "dashed")+ 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)+ settheme  +
  theme(legend.position = "none")
              
  
ggsave(plot = p , filename = sprintf("%s/D1_%s.pdf",out_dir, title), w = w, h = h)
p

In [ ]:
pair_stats <- pair_stats %>%
  mutate(
    cell_cell = case_when(
      intensity_before < GFP_thresh & intensity_after < GFP_thresh ~ "WT_WT",
      intensity_before > GFP_thresh & intensity_after > GFP_thresh ~ "GFP_GFP",
      intensity_before == "NaN" | intensity_after == "NaN" ~ NA, 
      TRUE ~ "GFP_WT" # Default case (mix of high/low)
    )
  )


In [ ]:
head(pair_stats)

In [ ]:
  # --- STEP 4: Merge back to original data ---
  df_final <- merged_df %>%
    select(-cell_cell)%>%
    left_join(pair_stats %>% select(sample_name,image, z, label_i, label_j, cell_cell), 
              by = c("sample_name" , "image", "z", "label_i", "label_j"))


In [ ]:
head(df_final)

In [ ]:
# by condition and state
plot_data <- df_final %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(sample_name, condition, state, image, cell_cell, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each state
  group_by(sample_name ,condition, state, distance_from_peak, cell_cell, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(sample_name, condition, state,      cell_cell,  channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  ) %>%
  filter(!is.na(cell_cell))

In [ ]:
w <- 6
h <- 2
title = "cell_cell_developed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid(  channel~ cell_cell, scales = "free_y") +
  settheme+
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/D2_%s.pdf",out_dir, title), w = w, h = h)
plot 